In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt
import os
import warnings
from IPython.display import display
from statsmodels.stats.inter_rater import fleiss_kappa

warnings.filterwarnings('ignore')

plt.style.use('default')
sns.set_palette("Set2")
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.notebook_repr_html', True)

In [ ]:
base_path = Path("../60_analyses/csv/qualitative/exp2/students")

BASE_PROJECT_PATH = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
hint_base_path = os.path.join(BASE_PROJECT_PATH, "20_experiments/60_analyses/csv/qualitative/exp2/hints")
output_base_path = os.path.join(BASE_PROJECT_PATH, "40_evaluation/exp2/qualitative")
output_tables_path = os.path.join(output_base_path, "tables")
output_plots_path = os.path.join(output_base_path, "plots")

for path in [output_tables_path, output_plots_path]:
    os.makedirs(path, exist_ok=True)

tables = {}
plots = {}

overlap_config = {
    "student_1": {
        "exp2a": range(1, 17),   # Questions 1-16
        "exp2b": range(1, 5),    # Questions 1-4
        "exp2c": []
    },
    "student_2": {
        "exp2a": range(11, 17),  # Questions 11-16  
        "exp2b": range(1, 9),    # Questions 1-8
        "exp2c": range(1, 7)     # Questions 1-6
    },
    "student_3": {
        "exp2a": [],
        "exp2b": range(5, 9),    # Questions 5-8
        "exp2c": range(1, 17)    # Questions 1-16
    }
}

In [ ]:
def load_student_data(student_id):
    student_path = base_path / f"student_{student_id}"
    data = {}
    
    for exp in ["exp2a", "exp2b", "exp2c"]:
        csv_path = student_path / f"{exp}.csv"
        if csv_path.exists():
            # Check overlap configuration first - skip if no questions assigned
            questions = overlap_config[f"student_{student_id}"][exp]
            if not questions:  # Skip if empty range
                print(f"Skipping {exp} for student_{student_id} - no questions assigned in overlap_config")
                continue
            
            # Try multiple encodings to handle German characters
            df = None
            for encoding in ['utf-8', 'latin-1', 'cp1252', 'iso-8859-1']:
                try:
                    df = pd.read_csv(csv_path, encoding=encoding)
                    break
                except UnicodeDecodeError:
                    continue
            
            if df is None:
                print(f"Error: Could not read {csv_path} with any encoding")
                continue
                
            df = df.drop('comments', axis=1)
            
            # Apply overlap configuration filter
            df['sample_id'] = df['sample_id'].astype(str)
            df['question_num'] = df['sample_id'].str.extract('(\d+)').astype(int).iloc[:, 0]
            
            # Filter to only include questions in overlap_config
            df_filtered = df[df['question_num'].isin(questions)]
            
            if len(df_filtered) == 0:
                print(f"Warning: No matching questions found for {exp} student_{student_id} with overlap_config")
                continue
            
            df_filtered['student'] = student_id
            df_filtered['experiment'] = exp
            data[exp] = df_filtered
            
            print(f"Loaded {exp} for student_{student_id}: {len(df_filtered)} questions (filtered by overlap_config: {list(questions)})")
    
    return data

def load_hint_data():
    hint_data = {}
    for exp in ["exp2a", "exp2b", "exp2c"]:
        hint_path = os.path.join(hint_base_path, f"{exp}_hints.csv")
        if os.path.exists(hint_path):
            # Try multiple encodings for hint files too
            hint_df = None
            for encoding in ['utf-8', 'latin-1', 'cp1252', 'iso-8859-1']:
                try:
                    hint_df = pd.read_csv(hint_path, encoding=encoding)
                    break
                except UnicodeDecodeError:
                    continue
            
            if hint_df is None:
                print(f"Error: Could not read {hint_path} with any encoding")
                continue
            
            # Handle different file structures
            if 'question_id' in hint_df.columns:
                # exp2a format: has explicit question_id column
                hint_df['sample_id'] = hint_df['question_id'].astype(str)
            elif 'bloom_original' in hint_df.columns:
                # exp2b and exp2c format: bloom_original contains the question_id
                hint_df['sample_id'] = hint_df['bloom_original'].astype(str)
            else:
                print(f"Warning: No question_id or bloom_original column found in {exp}_hints.csv")
                continue
                
            print(f"Loaded {exp}_hints.csv: {len(hint_df)} rows")
            hint_data[exp] = hint_df
    return hint_data

# Load all student data with strict overlap configuration
print("Loading student data with overlap configuration:")
for student, config in overlap_config.items():
    print(f"  {student}:")
    for exp, questions in config.items():
        if questions:
            print(f"    {exp}: Questions {list(questions)}")
        else:
            print(f"    {exp}: No questions assigned")

all_data = {}
for i in range(1, 4):
    # print(f"\n--- Loading student_{i} ---")
    all_data[f"student_{i}"] = load_student_data(i)

hint_data = load_hint_data()

In [ ]:
def combine_data():
    combined = []
    
    print("\n--- Combining data with overlap configuration ---")
    for student, experiments in all_data.items():
        print(f"\n{student}:")
        for exp, df in experiments.items():
            if not df.empty:
                # Add hint data (llm, question_type) 
                if exp in hint_data:
                    hint_df = hint_data[exp]
                    
                    # Determine which columns to merge based on what's available
                    merge_columns = ['sample_id', 'llm']
                    if 'question_type' in hint_df.columns:
                        merge_columns.append('question_type')
                    
                    # Merge on sample_id (both should be strings now)
                    df_merged = df.merge(hint_df[merge_columns], on='sample_id', how='left')
                    
                    # Verify overlap configuration is respected
                    questions_range = overlap_config[student][exp]
                    actual_questions = sorted(df_merged['question_num'].unique())
                    expected_questions = list(questions_range)
                    
                    print(f"  {exp}: {len(df_merged)} evaluations")
                    print(f"    Expected questions (overlap_config): {expected_questions}")
                    print(f"    Actual questions loaded: {actual_questions}")
                    
                    # Verify all questions are within expected range
                    unexpected = set(actual_questions) - set(expected_questions)
                    if unexpected:
                        print(f"    WARNING: Unexpected questions found: {list(unexpected)}")
                    
                    combined.append(df_merged)
                else:
                    print(f"  {exp}: {len(df)} evaluations (no hint data available)")
                    combined.append(df)
    
    if combined:
        return pd.concat(combined, ignore_index=True)
    return pd.DataFrame()

df_combined = combine_data()
print(f"\n=== FINAL SUMMARY ===")
print(f"Total evaluations: {len(df_combined)}")

if len(df_combined) > 0:
    print(f"\nBreakdown by student:")
    for student_id in sorted(df_combined['student'].unique()):
        student_data = df_combined[df_combined['student'] == student_id]
        print(f"  Student {student_id}: {len(student_data)} evaluations")
        for exp in ["exp2a", "exp2b", "exp2c"]:
            exp_data = student_data[student_data['experiment'] == exp]
            if len(exp_data) > 0:
                questions = sorted(exp_data['question_num'].unique())
                print(f"    {exp}: {len(exp_data)} evaluations, Questions: {questions}")
    
    print(f"\nBreakdown by experiment:")
    for exp in ["exp2a", "exp2b", "exp2c"]:
        exp_data = df_combined[df_combined['experiment'] == exp]
        print(f"  {exp}: {len(exp_data)} evaluations")
        if len(exp_data) > 0:
            students = sorted(exp_data['student'].unique())
            questions = sorted(exp_data['question_num'].unique())
            print(f"    Students: {students}")
            print(f"    Questions: {questions}")

    # Check LLM distribution
    if 'llm' in df_combined.columns:
        print(f"\nBreakdown by LLM:")
        llm_counts = df_combined.groupby('llm').size()
        for llm, count in llm_counts.items():
            print(f"  {llm}: {count} evaluations")
        
    if 'question_type' in df_combined.columns:
        print(f"\nBreakdown by question type:")
        qtype_counts = df_combined.groupby('question_type').size()
        for qtype, count in qtype_counts.items():
            print(f"  {qtype}: {count} evaluations")
    
    # Verify overlap configuration compliance
    print(f"\n=== OVERLAP CONFIGURATION VERIFICATION ===")
    for student_id in range(1, 4):
        student_key = f"student_{student_id}"
        if student_key in overlap_config:
            config = overlap_config[student_key]
            student_data = df_combined[df_combined['student'] == student_id]
            
            print(f"\nStudent {student_id}:")
            for exp in ["exp2a", "exp2b", "exp2c"]:
                expected_questions = list(config[exp]) if config[exp] else []
                exp_data = student_data[student_data['experiment'] == exp]
                actual_questions = sorted(exp_data['question_num'].unique()) if len(exp_data) > 0 else []
                
                status = "Y" if set(actual_questions) == set(expected_questions) else "N"
                print(f"  {exp}: {status}")
                print(f"    Expected: {expected_questions}")
                print(f"    Actual:   {actual_questions}")
                
                if set(actual_questions) != set(expected_questions):
                    missing = set(expected_questions) - set(actual_questions)
                    extra = set(actual_questions) - set(expected_questions)
                    if missing:
                        print(f"    Missing:  {list(missing)}")
                    if extra:
                        print(f"    Extra:    {list(extra)}")
    
    # Display sample of combined data
    print(f"\nSample of combined data:")
    print(df_combined[['student', 'experiment', 'sample_id', 'question_num', 'llm']].head(10))
else:
    print("No data loaded. Check if CSV files exist and contain data.")

In [ ]:
def calc_stats(df, metric):
    numeric_data = pd.to_numeric(df[metric], errors='coerce')
    return {
        'mean': numeric_data.mean(),
        'std': numeric_data.std(),
        'count': numeric_data.count()
    }

def overlap_analysis():
    results = {}
    
    print("=== OVERLAP ANALYSIS ===")
    print("Based on overlap_config, identifying questions evaluated by multiple students:")
    
    # Questions with multiple evaluations (overlaps) based on overlap_config
    overlap_questions = []
    overlap_details = {}
    
    # Check each question across all experiments
    for exp in ["exp2a", "exp2b", "exp2c"]:
        print(f"\n{exp}:")
        exp_data = df_combined[df_combined['experiment'] == exp]
        
        if len(exp_data) == 0:
            print(f"  No data for {exp}")
            continue
            
        # Count how many students evaluated each question
        question_counts = exp_data['sample_id'].value_counts()
        question_num_counts = exp_data['question_num'].value_counts()
        
        overlapping_questions = question_counts[question_counts > 1]
        
        print(f"  Total questions: {len(question_counts)}")
        print(f"  Questions with overlaps: {len(overlapping_questions)}")
        
        for question_id, count in overlapping_questions.items():
            overlap_questions.append(question_id)
            
            # Get students who evaluated this question
            question_data = exp_data[exp_data['sample_id'] == question_id]
            students = sorted(question_data['student'].unique())
            question_num = question_data['question_num'].iloc[0]
            
            overlap_details[question_id] = {
                'experiment': exp,
                'question_num': question_num,
                'students': students,
                'count': count
            }
            
            print(f"    Question {question_num} (ID: {question_id}): {count} evaluations by students {students}")
    
    # Verify overlap expectations based on overlap_config
    print(f"\n=== EXPECTED OVERLAPS BASED ON OVERLAP_CONFIG ===")
    expected_overlaps = {}
    
    for exp in ["exp2a", "exp2b", "exp2c"]:
        print(f"\n{exp}:")
        
        # Find questions that should have overlaps based on overlap_config
        question_to_students = {}
        
        for student_id in range(1, 4):
            student_key = f"student_{student_id}"
            questions = overlap_config[student_key][exp]
            
            for q_num in questions:
                if q_num not in question_to_students:
                    question_to_students[q_num] = []
                question_to_students[q_num].append(student_id)
        
        # Identify expected overlaps
        expected_exp_overlaps = {q: students for q, students in question_to_students.items() if len(students) > 1}
        expected_overlaps[exp] = expected_exp_overlaps
        
        if expected_exp_overlaps:
            print(f"  Expected overlapping questions:")
            for q_num, students in expected_exp_overlaps.items():
                print(f"    Question {q_num}: students {students}")
        else:
            print(f"  No overlapping questions expected")
    
    results['overlap_questions'] = overlap_questions
    results['total_overlaps'] = len(overlap_questions)
    results['overlap_details'] = overlap_details
    results['expected_overlaps'] = expected_overlaps
    
    return results

overlap_info = overlap_analysis()
print(f"\n=== SUMMARY ===")
print(f"Total questions with overlapping evaluations: {overlap_info['total_overlaps']}")

if overlap_info['total_overlaps'] > 0:
    print(f"\nOverlapping questions details:")
    for question_id, details in overlap_info['overlap_details'].items():
        print(f"  {details['experiment']} Q{details['question_num']}: {details['count']} evaluations by students {details['students']}")
else:
    print("No overlapping questions found in the loaded data.")

In [ ]:
# Define metrics and labels first
metrics = ['relevance', 'clarity', 'answerability', 'challenging', 'value', 'language', 'bloom_rating']
metric_labels = {
    'relevance': 'Relevance',
    'clarity': 'Clarity', 
    'answerability': 'Answerability',
    'challenging': 'Challenging',
    'value': 'Value',
    'language': 'Language',
    'bloom_rating': "Bloom's Level"
}

def inter_rater_reliability():
    reliability_results = {}
    
    print("=== INTER-RATER RELIABILITY ANALYSIS ===")
    print(f"Analyzing reliability for {len(overlap_info['overlap_questions'])} overlapping questions")
    
    for metric in metrics:
        agreements = []
        
        for q_id in overlap_info['overlap_questions']:
            q_data = df_combined[df_combined['sample_id'] == q_id]
            
            # Convert to numeric for all metrics consistently
            scores = pd.to_numeric(q_data[metric], errors='coerce').dropna()
            
            if len(scores) >= 2:
                # Calculate agreement (standard deviation as measure of disagreement)
                if len(scores.unique()) > 1:
                    agreements.append(scores.std())
                else:
                    agreements.append(0)  # Perfect agreement
        
        reliability_results[metric] = {
            'avg_std': np.mean(agreements) if agreements else 0,
            'num_comparisons': len(agreements),
            'min_std': np.min(agreements) if agreements else 0,
            'max_std': np.max(agreements) if agreements else 0
        }
    
    return reliability_results

reliability = inter_rater_reliability()

print(f"\nReliability Results:")
for metric, stats in reliability.items():
    metric_label = metric_labels.get(metric, metric.title())
    print(f"{metric_label}: avg std = {stats['avg_std']:.3f}, comparisons = {stats['num_comparisons']}")

# Reliability interpretation
print(f"\nReliability Interpretation (Lower std = Better agreement):")
for metric, stats in reliability.items():
    metric_label = metric_labels.get(metric, metric.title())
    level = ("Excellent" if stats['avg_std'] < 0.5 else 
            "Good" if stats['avg_std'] < 1.0 else 
            "Moderate" if stats['avg_std'] < 1.5 else 
            "Poor")
    print(f"  {metric_label}: {stats['avg_std']:.3f} std ({level})")

# Show detailed breakdown by experiment if there are overlaps
if overlap_info['total_overlaps'] > 0:
    print(f"\nDetailed overlap breakdown by experiment:")
    for exp in ["exp2a", "exp2b", "exp2c"]:
        exp_overlaps = [q for q, details in overlap_info['overlap_details'].items() if details['experiment'] == exp]
        if exp_overlaps:
            print(f"  {exp}: {len(exp_overlaps)} overlapping questions")
            for q_id in exp_overlaps:
                details = overlap_info['overlap_details'][q_id]
                print(f"    Question {details['question_num']}: students {details['students']}")
        else:
            print(f"  {exp}: No overlapping questions")

In [ ]:
def summary_stats():
    metrics = ['relevance', 'clarity', 'answerability', 'challenging', 'value', 'language', 'bloom_rating']
    
    summary = {}
    for exp in ["exp2a", "exp2b", "exp2c"]:
        exp_data = df_combined[df_combined['experiment'] == exp]
        summary[exp] = {}
        
        for metric in metrics:
            # Convert to numeric and calculate stats
            numeric_data = pd.to_numeric(exp_data[metric], errors='coerce')
            summary[exp][metric] = {
                'mean': numeric_data.mean(),
                'std': numeric_data.std(),
                'count': numeric_data.count()
            }
    
    return summary

stats = summary_stats()

In [ ]:
def create_seaborn_boxplot(data, x, y, ax, title, ylabel, xlabel, scale_range=None):
    colors = ['#66c2a5', '#fc8d62', '#8da0cb', '#e78ac3', '#a6d854', '#ffd92f']
    unique_vals = sorted(data[x].unique())
    palette = colors[:len(unique_vals)]
    
    sns.boxplot(data=data, x=x, y=y, ax=ax, palette=palette,
                medianprops={'color': 'black', 'linewidth': 2.5, 'linestyle': ':'})
    
    for i, val in enumerate(unique_vals):
        mean_val = data[data[x] == val][y].mean()
        ax.scatter(i, mean_val, color='red', marker='D', s=50, zorder=3, 
                  edgecolor='darkred', linewidth=1)
    
    label_mapping = {
        'exp2a': 'Type Only', 'exp2b': 'Bloom Only', 'exp2c': 'Type + Bloom'
    }
    
    current_labels = [tick.get_text() for tick in ax.get_xticklabels()]
    new_labels = [label_mapping.get(label, label) for label in current_labels]
    ax.set_xticklabels(new_labels, rotation=0)
    
    if scale_range:
        ax.set_ylim(scale_range)
        if scale_range == (0, 10):
            title += " (0-10 scale)"
    
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylabel(ylabel, fontsize=11, labelpad=15)
    ax.set_xlabel(xlabel, fontsize=11, labelpad=10)
    ax.grid(True, alpha=0.3)

# Create metric display labels for use in visualizations
metric_labels = {
    'relevance': 'Relevance',
    'clarity': 'Clarity', 
    'answerability': 'Answerability',
    'challenging': 'Challenging',
    'value': 'Value',
    'language': 'Language',
    'bloom_rating': "Bloom's Level"
}

# Experiment 2: Descriptive Statistics

Analysis of prompt engineering approaches for question generation.

In [ ]:
def convert_bloom_rating(row):
    """
    Convert bloom_rating based on experiment type and scoring rules.
    
    For exp2a: Convert bloom level to points
    - Creating (6): 10p
    - Evaluating (5): 8.5p  
    - Analyzing (4): 7p
    - Applying (3): 4.5p
    - Understanding (2): 3p
    - Remembering (1): 1.5p
    
    For exp2b and exp2c: Convert based on correctness vs expected level
    - Correct: 10p
    - Off by one: 5p
    - Otherwise: 0p
    """
    value = row['bloom_rating']
    experiment = row['experiment']
    sample_id = row['sample_id']
    
    if pd.isna(value) or value == -99:
        return np.nan
    
    # Convert to string for processing
    value_str = str(value).strip()
    
    # Handle tuple format like "(4;2)" - take the maximum
    if '(' in value_str and ')' in value_str:
        # Extract numbers from tuple format
        import re
        numbers = re.findall(r'\d+', value_str)
        if numbers:
            student_level = max([int(n) for n in numbers])
        else:
            return np.nan
    else:
        # Try to convert directly to number
        try:
            student_level = int(float(value_str))
        except (ValueError, TypeError):
            return np.nan
    
    if experiment == 'exp2a':
        # exp2a: Convert bloom level to points
        level_to_points = {
            6: 10.0,    # Creating
            5: 8.5,     # Evaluating
            4: 7.0,     # Analyzing
            3: 4.5,     # Applying
            2: 3.0,     # Understanding
            1: 1.5      # Remembering
        }
        return level_to_points.get(student_level, 0)
    
    else:  # exp2b and exp2c
        # Need to get expected bloom level from hint data
        if experiment in hint_data:
            hint_df = hint_data[experiment]
            expected_row = hint_df[hint_df['sample_id'] == sample_id]
            
            if len(expected_row) > 0:
                expected_level = expected_row['bloom_original'].iloc[0]
                try:
                    expected_level = int(expected_level)
                    if student_level == expected_level:
                        return 10.0  # Correct
                    elif abs(student_level - expected_level) == 1:
                        return 5.0   # Off by one
                    else:
                        return 0.0   # Otherwise
                except (ValueError, TypeError):
                    pass
        
        # 0 if no expected level found
        return 0.0

print("EXPERIMENT 2 - DESCRIPTIVE STATISTICS")
print("="*60)

metrics = ['relevance', 'clarity', 'answerability', 'challenging', 'value', 'language', 'bloom_rating']

# Create metric display labels
metric_labels = {
    'relevance': 'Relevance',
    'clarity': 'Clarity', 
    'answerability': 'Answerability',
    'challenging': 'Challenging',
    'value': 'Value',
    'language': 'Language',
    'bloom_rating': "Bloom's Level"
}

# Convert metrics to numeric, handling -99 and other non-numeric values
df_numeric = df_combined.copy()
for metric in metrics:
    if metric == 'bloom_rating':
        # Special handling for bloom_rating using row-wise function
        df_numeric[metric] = df_combined.apply(convert_bloom_rating, axis=1)
    else:
        df_numeric[metric] = pd.to_numeric(df_combined[metric], errors='coerce')

# Overall statistics
exp2_stats = df_numeric[metrics].describe().round(2)
# Rename columns for display
exp2_stats_display = exp2_stats.copy()
exp2_stats_display.columns = [metric_labels[col] for col in exp2_stats_display.columns]
tables['exp2_overall_stats'] = exp2_stats_display
print("\nOverall Statistics:")
display(exp2_stats_display)

# Statistics by Experiment 
exp2_exp_stats = df_numeric.groupby('experiment')[metrics].agg(['mean', 'std', 'median', 'count']).round(2)
# Rename columns for display
exp2_exp_stats_display = exp2_exp_stats.copy()
new_columns = []
for col in exp2_exp_stats_display.columns:
    metric, stat = col
    new_columns.append((metric_labels[metric], stat))
exp2_exp_stats_display.columns = pd.MultiIndex.from_tuples(new_columns)
tables['exp2_exp_stats'] = exp2_exp_stats_display
print("\nStatistics by Experiment:")
display(exp2_exp_stats_display)

# Experiment ranking
exp2_exp_means = df_numeric.groupby('experiment')[metrics].mean().round(2)
exp2_exp_overall = exp2_exp_means.mean(axis=1).sort_values(ascending=False)
tables['exp2_exp_ranking'] = exp2_exp_overall

print("\nOverall Experiment Ranking:")
exp_names = {'exp2a': 'Type Only', 'exp2b': 'Bloom Only', 'exp2c': 'Type + Bloom'}
for i, (exp, score) in enumerate(exp2_exp_overall.items(), 1):
    print(f"{i}. {exp_names[exp]}: {score:.2f}")

# Statistics by Student
exp2_student_stats = df_numeric.groupby('student')[metrics].agg(['mean', 'std', 'median', 'count']).round(2)
# Rename columns for display
exp2_student_stats_display = exp2_student_stats.copy()
new_columns = []
for col in exp2_student_stats_display.columns:
    metric, stat = col
    new_columns.append((metric_labels[metric], stat))
exp2_student_stats_display.columns = pd.MultiIndex.from_tuples(new_columns)
tables['exp2_student_stats'] = exp2_student_stats_display
print("\nStatistics by Student:")
display(exp2_student_stats_display)

# Statistics by LLM (if available)
if 'llm' in df_numeric.columns:
    # Filter out NaN LLMs for cleaner statistics
    df_llm = df_numeric.dropna(subset=['llm'])
    exp2_llm_stats = df_llm.groupby('llm')[metrics].agg(['mean', 'std', 'median', 'count']).round(2)
    # Rename columns for display
    exp2_llm_stats_display = exp2_llm_stats.copy()
    new_columns = []
    for col in exp2_llm_stats_display.columns:
        metric, stat = col
        new_columns.append((metric_labels[metric], stat))
    exp2_llm_stats_display.columns = pd.MultiIndex.from_tuples(new_columns)
    tables['exp2_llm_stats'] = exp2_llm_stats_display
    print("\nStatistics by LLM:")
    display(exp2_llm_stats_display)
    
    # LLM ranking
    exp2_llm_means = df_llm.groupby('llm')[metrics].mean().round(2)
    exp2_llm_overall = exp2_llm_means.mean(axis=1).sort_values(ascending=False)
    tables['exp2_llm_ranking'] = exp2_llm_overall
    
    print("\nOverall LLM Ranking:")
    for i, (llm, score) in enumerate(exp2_llm_overall.items(), 1):
        print(f"{i}. {llm.title()}: {score:.2f}")

# Statistics by Question Type (if available)
if 'question_type' in df_numeric.columns:
    df_qtype = df_numeric.dropna(subset=['question_type'])
    exp2_qtype_stats = df_qtype.groupby('question_type')[metrics].agg(['mean', 'std', 'median', 'count']).round(2)
    # Rename columns for display
    exp2_qtype_stats_display = exp2_qtype_stats.copy()
    new_columns = []
    for col in exp2_qtype_stats_display.columns:
        metric, stat = col
        new_columns.append((metric_labels[metric], stat))
    exp2_qtype_stats_display.columns = pd.MultiIndex.from_tuples(new_columns)
    tables['exp2_qtype_stats'] = exp2_qtype_stats_display
    print("\nStatistics by Question Type:")
    display(exp2_qtype_stats_display)

# Show bloom rating conversion summary
print("\nBloom Rating Conversion Summary:")
print("exp2a: Bloom level → Points (1=1.5, 2=3.0, 3=4.5, 4=7.0, 5=8.5, 6=10.0)")
print("exp2b/exp2c: Correctness vs expected → Points (Correct=10, Off by 1=5, Otherwise=0)")

# Show some example conversions
print("\nExample conversions:")
for exp in ['exp2a', 'exp2b', 'exp2c']:
    exp_data = df_combined[df_combined['experiment'] == exp].head(3)
    for _, row in exp_data.iterrows():
        original = row['bloom_rating']
        converted = convert_bloom_rating(row)
        print(f"  {exp} - '{original}' → {converted}")

# Check for problematic values in the original data
print("\nData Quality Check:")
for metric in metrics:
    if metric == 'bloom_rating':
        # Show original bloom_rating values that were converted
        original_values = df_combined[metric][df_combined[metric].notna()]
        non_standard = []
        for val in original_values.unique():
            val_str = str(val).strip()
            if '(' in val_str or not val_str.replace('.', '').replace('-', '').isdigit():
                non_standard.append(val)
        
        if non_standard:
            print(f"\nProcessed {metric_labels[metric]} values:")
            for val in non_standard[:10]:  # Show first 10
                print(f"  '{val}': processed")
    else:
        non_numeric = df_combined[metric][pd.to_numeric(df_combined[metric], errors='coerce').isna() & df_combined[metric].notna()]
        if len(non_numeric) > 0:
            print(f"\nNon-numeric values in {metric_labels[metric]}:")
            for val in non_numeric.unique()[:5]:  # Show first 5 unique non-numeric values
                print(f"  '{val}'")

In [ ]:
# Experiment Performance Visualization
fig, axes = plt.subplots(3, 3, figsize=(18, 16))
axes = axes.flatten()
plots['exp2_experiment_analysis'] = fig

for i, metric in enumerate(metrics):
    if metric == 'bloom_rating':
        # Use the already converted bloom ratings from df_numeric
        df_plot = df_numeric.copy()
        df_plot = df_plot.dropna(subset=[metric])
    else:
        df_plot = df_combined.copy()
        df_plot[metric] = pd.to_numeric(df_plot[metric], errors='coerce')
        df_plot = df_plot.dropna(subset=[metric])
    
    # Use proper metric label for display
    metric_display = metric_labels[metric]
    create_seaborn_boxplot(df_plot, 'experiment', metric, axes[i], 
                          metric_display, metric_display, 'Experiment', scale_range=(0, 10))

# Hide the last two empty subplots
axes[7].set_visible(False)
axes[8].set_visible(False)

plt.suptitle('Experiment 2: Performance across Prompt Engineering Approaches', 
             fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.subplots_adjust(top=0.93)
plt.show()

In [ ]:
# Student Performance Analysis
fig, axes = plt.subplots(3, 3, figsize=(18, 16))
axes = axes.flatten()
plots['exp2_student_analysis'] = fig

for i, metric in enumerate(metrics):
    if metric == 'bloom_rating':
        # Use the already converted bloom ratings from df_numeric
        df_plot = df_numeric.copy()
        df_plot = df_plot.dropna(subset=[metric])
    else:
        df_plot = df_combined.copy()
        df_plot[metric] = pd.to_numeric(df_plot[metric], errors='coerce')
        df_plot = df_plot.dropna(subset=[metric])
    
    colors = ['#66c2a5', '#fc8d62', '#8da0cb']
    
    sns.boxplot(data=df_plot, x='student', y=metric, ax=axes[i], palette=colors,
                medianprops={'color': 'black', 'linewidth': 2.5, 'linestyle': ':'})
    
    for j, student in enumerate(sorted(df_plot['student'].unique())):
        mean_val = df_plot[df_plot['student'] == student][metric].mean()
        axes[i].scatter(j, mean_val, color='red', marker='D', s=50, zorder=3, 
                       edgecolor='darkred', linewidth=1)
    
    axes[i].set_ylim(0, 10)
    metric_display = metric_labels[metric]
    axes[i].set_title(f'{metric_display} (0-10 scale)', fontsize=12, fontweight='bold')
    axes[i].set_ylabel(metric_display, fontsize=11, labelpad=15)
    axes[i].set_xlabel('Student', fontsize=11, labelpad=10)
    axes[i].grid(True, alpha=0.3)

# Hide the last two empty subplots
axes[7].set_visible(False)
axes[8].set_visible(False)

plt.suptitle('Experiment 2: Performance by Student Evaluator', 
             fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.subplots_adjust(top=0.93)
plt.show()

In [ ]:
# LLM Performance Analysis
if 'llm' in df_combined.columns:
    fig, axes = plt.subplots(3, 3, figsize=(18, 16))
    axes = axes.flatten()
    plots['exp2_llm_analysis'] = fig

    for i, metric in enumerate(metrics):
        if metric == 'bloom_rating':
            # Use the already converted bloom ratings from df_numeric
            df_plot = df_numeric.copy()
            df_plot = df_plot.dropna(subset=[metric])
        else:
            df_plot = df_combined.copy()
            df_plot[metric] = pd.to_numeric(df_plot[metric], errors='coerce')
            df_plot = df_plot.dropna(subset=[metric])
        
        colors = ['#66c2a5', '#fc8d62', '#8da0cb', '#e78ac3']
        
        sns.boxplot(data=df_plot, x='llm', y=metric, ax=axes[i], palette=colors,
                    medianprops={'color': 'black', 'linewidth': 2.5, 'linestyle': ':'})
        
        # Filter out NaN values before sorting
        valid_llms = df_plot['llm'].dropna().unique()
        for j, llm in enumerate(sorted(valid_llms)):
            mean_val = df_plot[df_plot['llm'] == llm][metric].mean()
            axes[i].scatter(j, mean_val, color='red', marker='D', s=50, zorder=3, 
                           edgecolor='darkred', linewidth=1)
        
        axes[i].set_ylim(0, 10)
        metric_display = metric_labels[metric]
        axes[i].set_title(f'{metric_display} (0-10 scale)', fontsize=12, fontweight='bold')
        axes[i].set_ylabel(metric_display, fontsize=11, labelpad=15)
        axes[i].set_xlabel('LLM', fontsize=11, labelpad=10)
        axes[i].grid(True, alpha=0.3)
        axes[i].tick_params(axis='x', rotation=45)

    # Hide the last two empty subplots
    axes[7].set_visible(False)
    axes[8].set_visible(False)

    plt.suptitle('Experiment 2: Performance by LLM', 
                 fontsize=16, fontweight='bold', y=0.98)
    plt.tight_layout()
    plt.subplots_adjust(top=0.93)
    plt.show()
else:
    print("LLM data not available for visualization")

In [ ]:
# Question Type Analysis
print("=== QUESTION TYPE ANALYSIS ===")

# Filter out NaN question types for analysis
df_qtype = df_numeric.dropna(subset=['question_type']).copy()
print(f"Analyzing {len(df_qtype)} questions with valid question type")

# Group by question type
qtype_counts = df_qtype['question_type'].value_counts()
print(f"Question type distribution: {dict(qtype_counts)}")

fig, axes = plt.subplots(3, 3, figsize=(18, 15))
axes = axes.flatten()

for i, metric in enumerate(metrics):
    metric_display = metric_labels.get(metric, metric)
    
    df_plot = df_qtype[['question_type', metric]].dropna()
    colors = ['#a6d854', '#ffd92f']
    
    sns.boxplot(data=df_plot, x='question_type', y=metric, ax=axes[i], palette=colors,
                medianprops={'color': 'black', 'linewidth': 2.5, 'linestyle': ':'})
    
    # Fix the sorting issue by filtering out NaN values
    valid_qtypes = df_plot['question_type'].dropna().unique()
    for j, qtype in enumerate(sorted(valid_qtypes)):
        mean_val = df_plot[df_plot['question_type'] == qtype][metric].mean()
        axes[i].scatter(j, mean_val, color='red', marker='D', s=50, zorder=3, 
                       edgecolor='darkred', linewidth=1)
    
    axes[i].set_title(f'{metric_display} by Question Type', fontsize=14, fontweight='bold')
    axes[i].set_xlabel('Question Type', fontsize=12)
    axes[i].set_ylabel(metric_display, fontsize=12)
    axes[i].grid(True, alpha=0.3)

# Remove empty subplots
for i in range(len(metrics), len(axes)):
    fig.delaxes(axes[i])

plt.tight_layout()
plt.savefig(f'{output_plots_path}/exp2_question_type_analysis.png', 
            dpi=300, bbox_inches='tight')
plt.show()

# Generate statistics
exp2_qtype_stats = {}
for metric in metrics:
    metric_data = df_qtype.dropna(subset=['question_type', metric])
    qtype_stats = metric_data.groupby('question_type')[metric].agg(['mean', 'std', 'count'])
    exp2_qtype_stats[metric] = qtype_stats

exp2_qtype_stats_display = {}
for metric in metrics:
    metric_display = metric_labels.get(metric, metric)
    exp2_qtype_stats_display[metric_display] = exp2_qtype_stats[metric]

print("\nQuestion Type Statistics:")
for metric_display, stats in exp2_qtype_stats_display.items():
    print(f"\n{metric_display}:")
    print(stats.round(3))

In [ ]:
# Heatmap Analysis
print("=== HEATMAP ANALYSIS ===")

# Create figure with subplots for different heatmaps
fig, ((ax0, ax1), (ax2, ax3)) = plt.subplots(2, 2, figsize=(20, 16))

# Get valid data only (filter out NaN values)
df_heatmap = df_numeric.dropna(subset=['experiment', 'llm']).copy()

experiments = sorted(df_heatmap['experiment'].unique())
llms = sorted(df_heatmap['llm'].dropna().unique())

# 1. Experiment vs Metrics Heatmap
exp_metrics_heatmap = pd.DataFrame(index=experiments, columns=[metric_labels.get(m, m) for m in metrics])
annot_matrix_exp = pd.DataFrame(index=experiments, columns=[metric_labels.get(m, m) for m in metrics])

for exp in experiments:
    for metric in metrics:
        exp_data = df_heatmap[df_heatmap['experiment'] == exp][metric].dropna()
        if len(exp_data) > 0:
            mean_val = exp_data.mean()
            std_val = exp_data.std()
            exp_metrics_heatmap.loc[exp, metric_labels.get(metric, metric)] = mean_val
            annot_matrix_exp.loc[exp, metric_labels.get(metric, metric)] = f"{mean_val:.1f}\n({std_val:.1f})"
        else:
            exp_metrics_heatmap.loc[exp, metric_labels.get(metric, metric)] = 0
            annot_matrix_exp.loc[exp, metric_labels.get(metric, metric)] = "N/A"

exp_labels = ['Type Only', 'Bloom Only', 'Type + Bloom']
sns.heatmap(exp_metrics_heatmap.astype(float), annot=annot_matrix_exp, fmt='', cmap='RdYlBu_r',
            center=exp_metrics_heatmap.astype(float).values.mean(), square=True, linewidths=0.5,
            cbar_kws={'shrink': 0.8, 'label': 'Average Score'}, annot_kws={'size': 10, 'weight': 'bold'},
            xticklabels=[metric_labels.get(m, m) for m in metrics], yticklabels=exp_labels, ax=ax0)

ax0.set_title('Mean Scores by Experiment\nValues: Mean (Std) | Scale: 0-10 points', 
             fontsize=14, fontweight='bold', pad=20)
ax0.set_xlabel('Evaluation Criteria', fontsize=12, fontweight='bold')
ax0.set_ylabel('Prompt Approach', fontsize=12, fontweight='bold')

# 2. LLM vs Experiment Heatmap
llm_exp_heatmap = pd.DataFrame(index=llms, columns=experiments)
annot_matrix_llm = pd.DataFrame(index=llms, columns=experiments)

for i, llm in enumerate(llms):
    for j, exp in enumerate(experiments):
        llm_exp_data = df_heatmap[(df_heatmap['llm'] == llm) & (df_heatmap['experiment'] == exp)]
        if len(llm_exp_data) > 0:
            # Calculate mean score across all metrics for this LLM-experiment combination
            scores = []
            for metric in metrics:
                metric_scores = llm_exp_data[metric].dropna()
                if len(metric_scores) > 0:
                    scores.extend(metric_scores.tolist())
            
            if scores:
                mean_val = np.mean(scores)
                llm_exp_heatmap.iloc[i, j] = mean_val
                annot_matrix_llm.iloc[i, j] = f"{mean_val:.1f}"
            else:
                llm_exp_heatmap.iloc[i, j] = 0
                annot_matrix_llm.iloc[i, j] = "N/A"
        else:
            llm_exp_heatmap.iloc[i, j] = 0
            annot_matrix_llm.iloc[i, j] = "N/A"

exp_labels = ['Type Only', 'Bloom Only', 'Type + Bloom']
llm_labels = [llm.title() for llm in llms]  # Now safe to use since we filtered NaN

sns.heatmap(llm_exp_heatmap.astype(float), annot=annot_matrix_llm, fmt='', cmap='RdYlBu_r',
            center=llm_exp_heatmap.astype(float).values.mean(), square=True, linewidths=0.5,
            cbar_kws={'shrink': 0.8, 'label': 'Average Score'}, annot_kws={'size': 10, 'weight': 'bold'},
            xticklabels=exp_labels, yticklabels=llm_labels, ax=ax1)

ax1.set_title('Average Scores by LLM vs Experiment\nValues: Mean (Std) | Scale: 0-10 points', 
             fontsize=14, fontweight='bold', pad=20)
ax1.set_xlabel('Prompt Approach', fontsize=12, fontweight='bold')
ax1.set_ylabel('Evaluator', fontsize=12, fontweight='bold')

# 3. Student vs Experiment Heatmap
students = sorted(df_heatmap['student'].unique())
student_exp_heatmap = pd.DataFrame(index=students, columns=experiments)
annot_matrix_student = pd.DataFrame(index=students, columns=experiments)

for i, student in enumerate(students):
    for j, exp in enumerate(experiments):
        student_exp_data = df_heatmap[(df_heatmap['student'] == student) & (df_heatmap['experiment'] == exp)]
        if len(student_exp_data) > 0:
            # Calculate mean score across all metrics for this student-experiment combination
            scores = []
            for metric in metrics:
                metric_scores = student_exp_data[metric].dropna()
                if len(metric_scores) > 0:
                    scores.extend(metric_scores.tolist())
            
            if scores:
                mean_val = np.mean(scores)
                student_exp_heatmap.iloc[i, j] = mean_val
                annot_matrix_student.iloc[i, j] = f"{mean_val:.1f}"
            else:
                student_exp_heatmap.iloc[i, j] = 0
                annot_matrix_student.iloc[i, j] = "N/A"
        else:
            student_exp_heatmap.iloc[i, j] = 0
            annot_matrix_student.iloc[i, j] = "N/A"

student_labels = [f'Student {s}' for s in students]
sns.heatmap(student_exp_heatmap.astype(float), annot=annot_matrix_student, fmt='', cmap='RdYlBu_r',
            center=student_exp_heatmap.astype(float).values.mean(), square=True, linewidths=0.5,
            cbar_kws={'shrink': 0.8, 'label': 'Average Score'}, annot_kws={'size': 10, 'weight': 'bold'},
            xticklabels=exp_labels, yticklabels=student_labels, ax=ax2)

ax2.set_title('Average Scores by Student vs Experiment\nValues: Mean (Std) | Scale: 0-10 points', 
             fontsize=14, fontweight='bold', pad=20)
ax2.set_xlabel('Prompt Approach', fontsize=12, fontweight='bold')
ax2.set_ylabel('Evaluator', fontsize=12, fontweight='bold')

# 4. Correlation Matrix
correlation_data = df_numeric[metrics].corr()
mask = np.triu(np.ones_like(correlation_data))

sns.heatmap(correlation_data, annot=True, fmt='.3f', cmap='RdBu_r', center=0,
            square=True, linewidths=0.5, cbar_kws={'shrink': 0.8, 'label': 'Correlation'},
            annot_kws={'size': 10, 'weight': 'bold'}, mask=mask,
            xticklabels=[metric_labels.get(m, m) for m in metrics], 
            yticklabels=[metric_labels.get(m, m) for m in metrics], ax=ax3)

ax3.set_title('Correlation Matrix Between Metrics\nUpper triangle masked for clarity', 
             fontsize=14, fontweight='bold', pad=20)

plt.tight_layout()
plt.savefig(f'{output_plots_path}/exp2_heatmap_analysis.png', 
            dpi=300, bbox_inches='tight')
plt.show()

print("Heatmap analysis completed and saved.")

# Inter-Student Reliability Analysis

Analysis of agreement between student evaluators on overlapping questions.

In [ ]:
# Fleiss' Kappa Analysis
print("\n" + "="*60)
print("FLEISS' KAPPA INTER-RATER RELIABILITY ANALYSIS")
print("="*60)

from statsmodels.stats.inter_rater import fleiss_kappa

def kappa_level(k):
    """Interpret Fleiss' Kappa value according to Landis & Koch (1977)"""
    if k < 0.2:
        return "Slight"
    elif k < 0.4:
        return "Fair" 
    elif k < 0.6:
        return "Moderate"
    elif k < 0.8:
        return "Substantial"
    else:
        return "Almost Perfect"

def ratings_to_fleiss_table(ratings_matrix):
    """Convert ratings matrix to Fleiss' Kappa format (items x categories)"""
    data = np.array(ratings_matrix)
    n_items, n_raters = data.shape
    table = np.zeros((n_items, 10))  # 1-10 rating scale
    
    for i in range(n_items):
        for j in range(n_raters):
            rating = data[i, j]
            if not np.isnan(rating) and 1 <= rating <= 10:
                table[i, int(rating) - 1] += 1
    
    return table

def calculate_fleiss_kappa_for_metric(metric, experiment_filter=None):
    """Calculate Fleiss' Kappa for a specific metric, optionally filtered by experiment"""
    
    metric_display = metric_labels.get(metric, metric)
    
    # Get relevant overlap questions
    relevant_questions = overlap_info['overlap_questions']
    if experiment_filter:
        relevant_questions = [q for q, details in overlap_info['overlap_details'].items() 
                            if details['experiment'] == experiment_filter]
    
    if len(relevant_questions) == 0:
        return np.nan, 0
    
    ratings_data = []
    
    for q_id in relevant_questions:
        if experiment_filter:
            q_data = df_combined[(df_combined['sample_id'] == q_id) & 
                               (df_combined['experiment'] == experiment_filter)]
        else:
            q_data = df_combined[df_combined['sample_id'] == q_id]
        
        if metric == 'bloom_rating':
            if experiment_filter:
                q_scores = df_numeric[(df_numeric['sample_id'] == q_id) & 
                                    (df_numeric['experiment'] == experiment_filter)][metric].dropna()
            else:
                q_scores = df_numeric[df_numeric['sample_id'] == q_id][metric].dropna()
        else:
            q_scores = pd.to_numeric(q_data[metric], errors='coerce').dropna()
        
        if len(q_scores) >= 2:
            scores_list = q_scores.tolist()
            # Ensure max 3 raters for experiment 2
            while len(scores_list) < 3:
                scores_list.append(np.nan)
            if len(scores_list) > 3:
                scores_list = scores_list[:3]
            ratings_data.append(scores_list)
    
    if len(ratings_data) < 2:
        return np.nan, len(ratings_data)
    
    try:
        ratings_matrix = np.array(ratings_data)
        valid_rows = ~np.isnan(ratings_matrix).all(axis=1)
        clean_ratings = ratings_matrix[valid_rows]
        
        if len(clean_ratings) == 0:
            return np.nan, 0
        
        # Fill NaN with row means
        for i in range(clean_ratings.shape[0]):
            row = clean_ratings[i]
            if np.isnan(row).any():
                row_mean = np.nanmean(row)
                if not np.isnan(row_mean):
                    clean_ratings[i] = np.where(np.isnan(row), row_mean, row)
        
        fleiss_table = ratings_to_fleiss_table(clean_ratings)
        kappa = fleiss_kappa(fleiss_table)
        
        return kappa, len(clean_ratings)
        
    except Exception as e:
        print(f"Error calculating Kappa for {metric_display}: {e}")
        return np.nan, len(ratings_data)

# 1. Overall Fleiss' Kappa for all metrics
print("\n=== OVERALL FLEISS' KAPPA BY METRIC ===")

overall_kappa_results = []
for metric in metrics:
    metric_display = metric_labels.get(metric, metric)
    kappa, n_questions = calculate_fleiss_kappa_for_metric(metric)
    
    overall_kappa_results.append({
        'Metric': metric_display,
        'Fleiss_Kappa': kappa,
        'Agreement_Level': kappa_level(kappa) if not np.isnan(kappa) else 'No data',
        'N_Questions': n_questions
    })

overall_fleiss_df = pd.DataFrame(overall_kappa_results)
tables['exp2_overall_fleiss_kappa'] = overall_fleiss_df
print("\nOverall Fleiss' Kappa Results:")
display(overall_fleiss_df.round(4))

# 2. Fleiss' Kappa by Sub-Experiment for all metrics
print("\n=== FLEISS' KAPPA BY SUB-EXPERIMENT ===")

experiments = ['exp2a', 'exp2b', 'exp2c']
exp_names = {'exp2a': 'Type Only', 'exp2b': 'Bloom Only', 'exp2c': 'Type + Bloom'}

# Create table with experiments as columns and metrics as rows
fleiss_by_exp_data = {}
for exp in experiments:
    fleiss_by_exp_data[exp_names[exp]] = []
    
    for metric in metrics:
        kappa, n_questions = calculate_fleiss_kappa_for_metric(metric, exp)
        fleiss_by_exp_data[exp_names[exp]].append(kappa)

# Create DataFrame
fleiss_by_exp_df = pd.DataFrame(fleiss_by_exp_data, 
                               index=[metric_labels.get(m, m) for m in metrics])
fleiss_by_exp_df.index.name = 'Metric'

tables['exp2_fleiss_by_experiment'] = fleiss_by_exp_df
print("\nFleiss' Kappa by Sub-Experiment:")
display(fleiss_by_exp_df.round(4))

# Create interpretation table
fleiss_by_exp_interpretation = fleiss_by_exp_df.copy()
for col in fleiss_by_exp_interpretation.columns:
    fleiss_by_exp_interpretation[col] = fleiss_by_exp_interpretation[col].apply(
        lambda x: kappa_level(x) if not np.isnan(x) else 'No data'
    )

tables['exp2_fleiss_interpretation'] = fleiss_by_exp_interpretation
print("\nAgreement Level Interpretation:")
display(fleiss_by_exp_interpretation)

print("\nFleiss' Kappa Interpretation (Landis & Koch, 1977):")
print("  < 0.20: Slight agreement")
print("  0.20-0.40: Fair agreement")
print("  0.40-0.60: Moderate agreement") 
print("  0.60-0.80: Substantial agreement")
print("  > 0.80: Almost perfect agreement")

In [ ]:
def save_all_results():
    print("Saving results...")
    
    # Use student prefix for exp2
    prefix = "student_"
    
    tables_saved = 0
    for table_name, table_data in tables.items():
        csv_path = os.path.join(output_tables_path, f"{prefix}{table_name}.csv")
        table_data.to_csv(csv_path)
        tables_saved += 1
        print(f"Saved table: {prefix}{table_name}.csv")
    
    plots_saved = 0
    for plot_name, plot_fig in plots.items():
        png_path = os.path.join(output_plots_path, f"{prefix}{plot_name}.png")
        plot_fig.savefig(png_path, dpi=300, bbox_inches='tight')
        plots_saved += 1
        print(f"Saved plot: {prefix}{plot_name}.png")
    
    print(f"\nResults saved:")
    print(f"  {tables_saved} tables saved to: {output_tables_path}")
    print(f"  {plots_saved} plots saved to: {output_plots_path}")
    print(f"  Output location: {output_base_path}")
    print(f"  File prefix: {prefix}")

save_all_results()

print(f"\nData successfully analyzed and exported!")
print(f"Total questions analyzed: {len(df_combined)}")
print(f"Students: {sorted(df_combined['student'].unique())}")
print(f"Experiments: {sorted(df_combined['experiment'].unique())}")
print(f"Metrics analyzed: {metrics}")

print("Only the overlapping questions per student/experiment were included in the analysis:")
for student_key, student_config in overlap_config.items():
    student_num = student_key.split('_')[1]
    print(f"\nStudent {student_num}:")
    for exp, questions in student_config.items():
        if questions:
            print(f"  {exp}: Questions {list(questions)}")
        else:
            print(f"  {exp}: No questions assigned")

# Final Fleiss' Kappa summary
print(f"\n=== FINAL FLEISS' KAPPA SUMMARY ===")
if 'overall_fleiss_df' in locals() and not overall_fleiss_df.empty:
    valid_kappas = overall_fleiss_df.dropna(subset=['Fleiss_Kappa'])
    if len(valid_kappas) > 0:
        avg_kappa = valid_kappas['Fleiss_Kappa'].mean()
        print(f"Average Fleiss' Kappa: {avg_kappa:.4f} ({kappa_level(avg_kappa)})")
        print(f"Metrics with valid Kappa: {len(valid_kappas)}/{len(overall_fleiss_df)}")
        
        best_metric = valid_kappas.loc[valid_kappas['Fleiss_Kappa'].idxmax(), 'Metric']
        best_kappa = valid_kappas['Fleiss_Kappa'].max()
        print(f"Best inter-rater agreement: {best_metric} (κ = {best_kappa:.4f})")
        
        # Agreement level distribution
        agreement_counts = valid_kappas['Agreement_Level'].value_counts()
        print(f"\nAgreement Level Distribution:")
        for level, count in agreement_counts.items():
            print(f"  {level}: {count} metrics")
else:
    print("No Fleiss' Kappa results available")